# Material-Level Analysis: Engineering the Perovskite Chemical Space

## 1. Abstract
This notebook serves as the foundational material analysis for the $ABX_3$ perovskite system. We move beyond raw chemical formulas to engineer physics-informed descriptors. These descriptors map the discrete chemical space into a continuous numerical manifold suitable for Gradient Boosted Decision Trees (GBDTs).

## 2. Theoretical Framework: The Physics-to-ML Mapping

### 2.1. Goldschmidt Tolerance Factor ($t$): The Geometric Gatekeeper
The stability of the $ABX_3$ perovskite structure is governed by the packing efficiency of the constituent ions. The Goldschmidt Tolerance Factor is defined as:

$$
t = \frac{r_A + r_X}{\sqrt{2}(r_B + r_X)}
$$

*   **Physics Role:** It determines the structural phase. Ideally, $t \approx 1.0$ for cubic structures. If $t < 0.9$, the $BX_6$ octahedra tilt to fill the space, reducing symmetry and altering the electronic properties. Values $t > 1.05$ typically lead to hexagonal 2D phases.
*   **ML Connection:** CatBoost treats $t$ as a **Structural Prior**. By splitting on $t$, the model can partition the dataset into 'Stable Cubic' vs. 'Tilted Orthorhombic' vs. '2D Layered' regimes. This allows the model to learn phase-specific behavior without explicit crystal symmetry labels.

### 2.2. Octahedral Factor ($\mu$): Coordination Stability
While $t$ describes the global packing, $\mu$ describes the local stability of the $B$-site cation within its $X$-site octahedral cage:

$$
\mu = \frac{r_B}{r_X}
$$

*   **Physics Role:** It dictates whether the $BX_6$ octahedra can physically form. If $\mu$ is too small ($< 0.41$), the $B$-site ion 'rattles' within the cage, leading to structural instability.
*   **ML Connection:** This feature serves as a **Feasibility Filter**. In the inverse design engine, the optimizer uses $\mu$ to prune 'unphysical' compositions that would mathematically predict high PCE but physically cannot crystallize.

### 2.3. Stoichiometric Weighted Radii ($r_{eff}$)
For mixed-cation systems (e.g., $Cs_x FA_{1-x} Pb I_3$), we use the effective ionic radius based on stoichiometry:

$$
r_{A,eff} = \sum_{i} x_i r_i
$$

*   **Physics Role:** Larger cations ($FA^+$) expand the lattice, while smaller ones ($Cs^+$) contract it, directly shifting the electronic energy levels via orbital overlap changes (Band Gap tuning).
*   **ML Connection:** This provides a **Continuous Manifold** for the model to interpolate between discrete elements. It transforms the categorical problem into a regression on a physical property.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_white"

df_p = pd.read_parquet('../data/enriched_perovskites.parquet')
print(f"Dataset enriched with {len(df_p.columns)} features for {len(df_p)} materials.")

## 3. Data Distribution: Mapping the Chemical Coverage
We visualize the cardinality of our categorical features. A high concentration of data in specific regions (e.g., Lead-based perovskites) indicates experimental bias which the ML model must navigate.

$$
N_{samples} = \sum_{i} count(Element_i)
$$

In [ ]:
fig1 = px.bar(df_p['A_1'].value_counts().reset_index(), x='A_1', y='count', 
             title="Cardinality of A-site Cations", 
             labels={'A_1': 'Primary Cation', 'count': 'Number of Samples'})
fig1.show()

fig2 = px.pie(df_p, names='dimension', title="Material Dimensionality Distribution (3D vs 2D)")
fig2.show()

## 4. Descriptor Correlation: Physical Redundancy
We analyze the Pearson correlation matrix to identify physical dependencies. 

$$
\rho_{X,Y} = \frac{\text{cov}(X,Y)}{\sigma_X \sigma_Y}
$$

A high correlation between $r_A$ and $t$ is expected, as $t$ is mathematically derived from $r_A$. The ML model (CatBoost) handles this multicollinearity well via its decision-tree structure, but redundant features should be monitored.

In [ ]:
cols = ['r_A', 'en_A', 'mass_A', 'r_C', 'en_C', 'mass_C', 'tolerance_factor', 'octahedral_factor', 'band_gap']
corr = df_p[cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.index,
    y=corr.columns,
    colorscale='RdBu_r',
    zmin=-1, zmax=1
))
fig.update_layout(
    title="Correlation Matrix of Physics-Informed Descriptors",
    xaxis_title="Input Feature",
    yaxis_title="Target/Feature",
    width=800, height=700
)
fig.show()

## 5. The Stability Manifold: $t$ vs. $\mu$
The structural stability of perovskites is localized in a specific region of the $t$-$\mu$ space. The density of experimental points shows the 'Stable 3D Manifold' ($t \in [0.82, 1.03]$).

**ML Interaction:** The model learns that outside this manifold, the Band Gap and Stability metrics behave drastically differently, as the crystal symmetry changes from cubic to lower-order or layered systems.

In [ ]:
fig = px.density_contour(
    df_p, 
    x='tolerance_factor', 
    y='octahedral_factor', 
    title="Experimental Density in the Stability Manifold",
    labels={'tolerance_factor': 'Tolerance Factor (t)', 'octahedral_factor': 'Octahedral Factor (μ)'}
)
fig.add_vrect(x0=0.82, x1=1.03, fillcolor="green", opacity=0.05, annotation_text="Stable 3D Manifold")
fig.update_layout(
    xaxis_title="Tolerance Factor (Goldschmidt)",
    yaxis_title="Octahedral Factor (rB/rX)",
    width=900, height=600
)
fig.show()

## 6. Electronic Manifold: Band Gap vs. Anion Descriptors
The Band Gap ($E_g$) in perovskites is primarily determined by the Halide ($X$) p-orbitals. We visualize how the model captures the trend where increasing electronegativity ($\chi_X$) widens the gap.

$$
E_g \approx \Delta E(\text{Halide p-orbitals}) + \text{Geometric Corrections}
$$

In [ ]:
fig = px.scatter_3d(
    df_p, 
    x='en_C', 
    y='r_C', 
    z='band_gap', 
    color='C_1', 
    title="3D Electronic Manifold: Band Gap vs. Anion Descriptors",
    labels={'en_C': 'Electronegativity', 'r_C': 'Radius (Å)', 'band_gap': 'Eg (eV)'}
)
fig.update_layout(width=1000, height=700)
fig.show()

## 7. Scientific Limitations and ML Challenges

### 7.1. Limitations of Geometric Descriptors
1.  **Dynamic Stabilization:** Rigid ionic radii ignore the dynamic rotational entropy of organic cations ($MA, FA$), which can stabilize structures outside the classic $t$ range.
2.  **Orbital Physics:** Geometric factors cannot capture relativistic effects (e.g., spin-orbit coupling in Pb-based systems) which significantly shift Band Gaps.
3.  **Local Distortions:** The tolerance factor is an average; it fails to describe local octahedral tilting or distortion in mixed-cation systems.

### 7.2. ML Reliability & Data Bias
1.  **Selection Bias:** Experimental data in the literature is biased towards 'successful' materials ($PCE > 10\%$). The ML model may struggle to predict 'failure' in unstable regions accurately.
2.  **Extrapolation Risk:** The model is reliable within the experimental convex hull of chemical compositions. Predicting properties for entirely new elements (e.g., $Ge$-based) is highly uncertain due to a lack of training overlap.